# Pretrained Siamese ResNet-18

This notebook trains a **siamese change-detection network with an ImageNet-pretrained ResNet-18 encoder** and evaluates it on held-out disasters. It is the pretrained counterpart to `train_scratch_cnn.ipynb`: both read the same extracted pre/post building patches and the same split assignment, so comparing the two isolates the effect of ImageNet pretraining.

Training is two-phase:a frozen-backbone head warm-up, then fine-tuning of the upper ResNet blocks, with the model selected on validation macro-F1 and evaluated once on the in-distribution and wildfire test sets.


## Task 1: Setup and Shared Data

Check the runtime, fix every routine choice in one configuration cell, and load the pre/post patch arrays shared with the from-scratch notebook. The preflight validates array layout, dtypes, labels, and split names before any weights are built.

### 1.1 Environment and imports

In [ ]:
import importlib.util

REQUIRED = {
    'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib',
    'torch': 'torch', 'torchvision': 'torchvision', 'sklearn': 'scikit-learn',
}
missing = [package for module, package in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ImportError('Missing packages: ' + ', '.join(missing) +
                      '\nInstall the local requirements and PyTorch; see docs/local_setup.md.')

import json, random, time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.metrics import (confusion_matrix, f1_score, precision_recall_fscore_support,
                             roc_auc_score)

print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

### 1.2 Run configuration

Paths, seed, and the training budget for both phases. `EPOCHS_HEAD` warms up the classifier head on a frozen backbone; `EPOCHS_FT` then fine-tunes the upper encoder blocks at a 10x lower learning rate. Early stopping on validation macro-F1 normally ends training well before the ceiling.

In [ ]:
# Configuration. Change only this cell for ordinary runs.
PROJECT_ROOT_OVERRIDE = None  # e.g. r'D:\projects\satellite-disaster-damage-mapping'
PATCH_DIR_OVERRIDE = None
OUTPUT_DIR_OVERRIDE = None

BATCH_SIZE = 128
SEED = 42
EPOCHS_HEAD = 5        # phase 1: frozen backbone, train the head only
EPOCHS_FT = 45         # phase 2: fine-tune layer3 + layer4
LR_HEAD = 1e-3
LR_FT = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 8           # early stop on val macro-F1

CLASSES = ['no-damage', 'minor-damage', 'major-damage', 'destroyed']
SHORT = ['none', 'minor', 'major', 'destr']
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], np.float32)


def find_project_root():
    candidates = [Path(PROJECT_ROOT_OVERRIDE)] if PROJECT_ROOT_OVERRIDE else []
    here = Path.cwd().resolve()
    candidates += [here, *here.parents]
    for candidate in candidates:
        if (candidate / '.git').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    return None


PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the project root. Run from the repo or set PROJECT_ROOT_OVERRIDE.')
PATCH_DIR = Path(PATCH_DIR_OVERRIDE) if PATCH_DIR_OVERRIDE else PROJECT_ROOT / 'work-pretrained' / 'patches'
OUTPUT_DIR = Path(OUTPUT_DIR_OVERRIDE) if OUTPUT_DIR_OVERRIDE else PROJECT_ROOT / 'work-pretrained'
print('patches:', PATCH_DIR)
print('outputs:', OUTPUT_DIR)

### 1.3 Shared patch inputs

Load `work-pretrained/patches/{manifest.csv, pre.npy, post.npy}` and validate them. These are the exact arrays the from-scratch notebook uses, which is what keeps the two experiments comparable. If they are missing, run `notebooks/data_cleaning_eda.ipynb` first.

In [ ]:
# Input preflight: fail before building a model or writing any output.
manifest_path, pre_path, post_path = (PATCH_DIR / 'manifest.csv', PATCH_DIR / 'pre.npy', PATCH_DIR / 'post.npy')
missing_inputs = [str(p) for p in (manifest_path, pre_path, post_path) if not p.is_file()]
if missing_inputs:
    raise FileNotFoundError(
        'Missing patch inputs:\n  ' + '\n  '.join(missing_inputs) +
        '\n\nThe pre/post patch arrays are produced by the data-preparation pipeline '
        '(notebooks/data_cleaning_eda.ipynb). This notebook only trains and evaluates on them.')

manifest = pd.read_csv(manifest_path)
required_columns = {'row', 'label', 'split'}
if absent := required_columns - set(manifest.columns):
    raise ValueError(f'manifest.csv is missing columns: {sorted(absent)}')
if 'status' in manifest.columns:                       # keep only successfully extracted patches
    manifest = manifest[manifest.status == 1].copy()

expected_splits = {'train', 'val', 'test_id', 'test_ood'}
found_splits = set(manifest['split'].dropna().unique())
if found_splits != expected_splits:
    raise ValueError(f'Expected splits {sorted(expected_splits)}, found {sorted(found_splits)}')
manifest['row'] = pd.to_numeric(manifest['row'], errors='raise').astype(np.int64)
manifest['label'] = pd.to_numeric(manifest['label'], errors='raise').astype(np.int64)
if not manifest['label'].between(0, len(CLASSES) - 1).all():
    raise ValueError(f'labels must be in [0, {len(CLASSES) - 1}]')

pre_array, post_array = np.load(pre_path, mmap_mode='r'), np.load(post_path, mmap_mode='r')
if pre_array.shape != post_array.shape or pre_array.ndim != 4 or pre_array.shape[-1] != 3:
    raise ValueError(f'Expected matching (N, H, W, 3) arrays, got {pre_array.shape} and {post_array.shape}')
if pre_array.dtype != np.uint8 or post_array.dtype != np.uint8:
    raise TypeError(f'Expected uint8 arrays, got {pre_array.dtype} and {post_array.dtype}')
if manifest['row'].min() < 0 or manifest['row'].max() >= len(pre_array) or manifest['row'].duplicated().any():
    raise ValueError('manifest row values must be unique and within the patch-array bounds')

print('Preflight passed:', pre_array.shape, pre_array.dtype)
display(manifest.groupby('split').size().reindex(['train', 'val', 'test_id', 'test_ood']).rename('patches').to_frame())

## Task 2: Data Pipeline and Model


### 2.1 Patch-pair dataset and augmentation

Each example is a pair of co-registered 64x64 RGB patches, the same building before and after the disaster. `PatchPairDataset` reads both from the memory-mapped arrays and applies **ImageNet normalisation**, matching the statistics the pretrained encoder expects.

Training-only augmentation (flips, 90-degree rotations, mild brightness jitter) is applied **identically to both patches in a pair** so the change signal is preserved. Validation and test patches are never augmented.

In [ ]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class PatchPairDataset(Dataset):
    """Pre/post 64x64 patch pairs from the memory-mapped arrays.

    Augmentation is applied IDENTICALLY to both images of a pair: independent jitter per
    branch would manufacture a change signal where none exists.
    """

    def __init__(self, frame, patch_dir=None, augment=False):
        self.rows = frame.row.to_numpy(np.int64)
        self.labels = frame.label.to_numpy(np.int64)
        self.patch_dir = Path(patch_dir) if patch_dir is not None else PATCH_DIR
        self.augment = augment
        self._pre = self._post = None            # opened lazily

    def __len__(self):
        return len(self.rows)

    def arrays(self):
        if self._pre is None:
            self._pre = np.load(self.patch_dir / 'pre.npy', mmap_mode='r')
            self._post = np.load(self.patch_dir / 'post.npy', mmap_mode='r')
        return self._pre, self._post

    def normalize(self, image):
        image = (image.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
        return torch.from_numpy(np.ascontiguousarray(image.transpose(2, 0, 1)))

    def __getitem__(self, index):
        pre_data, post_data = self.arrays()
        row = self.rows[index]
        pre = np.asarray(pre_data[row])
        post = np.asarray(post_data[row])
        if self.augment:
            if np.random.rand() < 0.5:                       # horizontal flip
                pre, post = pre[:, ::-1], post[:, ::-1]
            if np.random.rand() < 0.5:                       # vertical flip
                pre, post = pre[::-1], post[::-1]
            turns = np.random.randint(4)                     # 90-degree rotation
            if turns:
                pre, post = np.rot90(pre, turns), np.rot90(post, turns)
            if np.random.rand() < 0.3:                       # shared brightness jitter
                gain = np.float32(np.random.uniform(0.85, 1.15))
                pre = np.clip(pre.astype(np.float32) * gain, 0, 255).astype(np.uint8)
                post = np.clip(post.astype(np.float32) * gain, 0, 255).astype(np.uint8)
        return self.normalize(pre), self.normalize(post), int(self.labels[index])

### 2.2 Siamese pretrained ResNet-18 architecture

**Encoder**: one ImageNet ResNet-18, shared across `pre` and `post`:
- `fc` replaced with `Identity`, giving a 512-d embedding per patch
- weights from `ResNet18_Weights.IMAGENET1K_V1` (falls back to random init only if the download fails)

**Change head:**
- combine the embeddings as `[f_pre, f_post, |f_pre - f_post|]` (1536-d) so the head sees appearance *and* the explicit change term
- `Linear(1536 -> 256)` -> `BatchNorm1d` -> ReLU -> `Dropout(0.4)` -> `Linear(256 -> 4)`

**Freezing strategy** (`set_trainable`):
- **Phase 1** freezes the whole encoder and trains only the head, so the randomly initialised head does not push large gradients into good pretrained weights.
- **Phase 2** unfreezes `layer3` and `layer4` at a 10x lower LR. Early layers (edges, textures) transfer almost unchanged; the late semantic layers are what must be repurposed. `layer1`/`layer2` stay frozen, which also regularises.

ResNet-18 is chosen over deeper backbones because at 64 px input depth stops paying off, and because its capacity is close to the from-scratch CNN's, so any measured difference isolates *pretraining* rather than model size.

In [ ]:
class SiamesePretrained(nn.Module):
    """Shared ImageNet ResNet-18 encoder + a small head over [f_pre, f_post, |f_pre - f_post|]."""

    def __init__(self, num_classes=len(CLASSES), dropout=0.4):
        super().__init__()
        try:
            net = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
            self.pretrained = True
        except Exception as error:                       # no internet on the first run
            print('!! could not fetch ImageNet weights:', error)
            net = resnet18(weights=None)
            self.pretrained = False
        self.feat_dim = net.fc.in_features               # 512 for ResNet-18
        net.fc = nn.Identity()
        self.encoder = net
        self.head = nn.Sequential(
            nn.Linear(self.feat_dim * 3, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def set_trainable(self, unfreeze=()):
        """Freeze the whole encoder, then re-enable the named blocks. The head stays trainable."""
        for parameter in self.encoder.parameters():
            parameter.requires_grad = False
        for name in unfreeze:
            for parameter in getattr(self.encoder, name).parameters():
                parameter.requires_grad = True
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f'  trainable {trainable / 1e6:.2f}M / {total / 1e6:.2f}M params '
              f'| unfrozen encoder blocks: {list(unfreeze) or "none"}')

    def forward(self, pre, post):
        pre_features = self.encoder(pre)
        post_features = self.encoder(post)
        change_features = torch.abs(pre_features - post_features)
        return self.head(torch.cat([pre_features, post_features, change_features], dim=1))

### 2.3 Runtime: device, seeds, and checkpoint paths

Seed every RNG, pick the device, enable mixed precision on CUDA, and create the output and checkpoint folders.

In [ ]:
set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
amp_enabled = device.type == 'cuda'
if amp_enabled:
    torch.backends.cudnn.benchmark = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_dir = OUTPUT_DIR / 'checkpoints'
checkpoint_dir.mkdir(exist_ok=True)
best_path = checkpoint_dir / 'best.pt'

print(f'device: {device}')
print(f'CUDA mixed precision: {amp_enabled}')
if amp_enabled:
    print('GPU:', torch.cuda.get_device_name(0))
print(f'outputs: {OUTPUT_DIR}')

### 2.4 Dataset and loader objects

One `PatchPairDataset` per split (augmentation on for `train` only), each wrapped in a `DataLoader`. The sanity check pulls one training batch and prints the tensor shapes.

In [ ]:
NUM_WORKERS = 0  # DataLoader workers use spawn on Windows and can hang in Jupyter; the patches are
                 # a small memmap, so loading is not the bottleneck. Raise to 4 on Linux/macOS.

datasets = {
    split: PatchPairDataset(manifest[manifest.split.eq(split)], PATCH_DIR, augment=split == 'train')
    for split in ['train', 'val', 'test_id', 'test_ood']
}
loaders = {
    split: DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=split == 'train',
                      num_workers=NUM_WORKERS, pin_memory=amp_enabled, drop_last=False)
    for split, dataset in datasets.items()
}
print('Dataset sizes:')
for split, dataset in datasets.items():
    print(f'  {split}: {len(dataset):,} patches')

pre_batch, post_batch, label_batch = next(iter(loaders['train']))
print(f'\npre batch {tuple(pre_batch.shape)} | post batch {tuple(post_batch.shape)} '
      f'| labels {tuple(label_batch.shape)}')

### 2.5 Class-weighted loss weights

The damage classes are heavily imbalanced and most patches are `no-damage`. Inverse-frequency weights computed from the **training labels only** are passed to cross-entropy so the rarer, more severe classes are not ignored.

In [ ]:
train_labels = datasets['train'].labels
class_counts = np.bincount(train_labels, minlength=len(CLASSES)).astype(float)
class_weights = class_counts.sum() / (len(CLASSES) * np.maximum(class_counts, 1))

display(pd.DataFrame({
    'class': CLASSES,
    'train_patches': class_counts.astype(int),
    'share_percent': (100 * class_counts / class_counts.sum()).round(2),
    'loss_weight': class_weights.round(3),
}))
print('Weights are inverse-frequency and use training labels only.')

## Task 3: Training

Two phases, both optimised with AdamW under a per-phase cosine schedule and mixed precision. After every epoch the model is scored on the validation split; the best validation macro-F1 checkpoint is kept, and each phase stops early once macro-F1 stalls for `PATIENCE` epochs.

### 3.1 Instantiate the model

Build `SiamesePretrained`, move it to the device, and confirm the ImageNet weights loaded.

In [ ]:
set_seed(SEED)
model = SiamesePretrained().to(device)
parameter_count = sum(p.numel() for p in model.parameters())
print('ImageNet weights loaded:', model.pretrained)
if not model.pretrained:
    print('!! Without pretrained weights this notebook does not answer its research question.')
print(f'Parameter count: {parameter_count:,}')

### 3.2 Loss, optimizer, and scheduler

**Loss:** cross-entropy with the inverse-frequency class weights from Section 2.5.

**Optimizer:** AdamW, rebuilt per phase over only the currently-trainable parameters
- phase 1 (head): lr `1e-3`
- phase 2 (fine-tune): lr `1e-4`
- weight decay `1e-4`

**Schedule:** `CosineAnnealingLR` over each phase's epoch count.

**Mixed precision:** `GradScaler` + `autocast` on CUDA; the helper absorbs the API change in Torch 2.4.

In [ ]:
criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))


def make_scaler(enabled):
    """GradScaler across torch versions (the signature changed in 2.4)."""
    try:
        return torch.amp.GradScaler('cuda', enabled=enabled)
    except (TypeError, AttributeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def autocast():
    return torch.autocast(device_type=device.type, enabled=amp_enabled)


scaler = make_scaler(amp_enabled)
print(f'phase 1 (head)      : {EPOCHS_HEAD} epochs @ lr {LR_HEAD}')
print(f'phase 2 (fine-tune) : {EPOCHS_FT} epochs @ lr {LR_FT}  (layer3 + layer4)')
print(f'weight decay {WEIGHT_DECAY} | patience {PATIENCE} | batch size {BATCH_SIZE}')

### 3.3 Evaluation helper

Reused for the per-epoch validation pass and the final held-out evaluation. Returns mean loss plus stacked true labels, predictions, and softmax probabilities.

In [ ]:
@torch.no_grad()
def evaluate(loader):
    """Return (mean_loss, y_true, y_pred, y_prob) for a loader."""
    model.eval()
    total_loss, count = 0.0, 0
    truths, predictions, probabilities = [], [], []
    for pre, post, labels in loader:
        pre = pre.to(device, non_blocking=True)
        post = post.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with autocast():
            logits = model(pre, post)
            loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        count += labels.size(0)
        truths.append(labels.cpu().numpy())
        predictions.append(logits.argmax(1).cpu().numpy())
        probabilities.append(torch.softmax(logits.float(), 1).cpu().numpy())
    return (total_loss / max(count, 1), np.concatenate(truths),
            np.concatenate(predictions), np.concatenate(probabilities))


print('Evaluation helper ready.')

### 3.4 Two-phase training loop

`run_phase()` runs one phase's epoch loop: forward/backward under autocast, `scheduler.step()`, then a validation pass. It writes `best.pt` whenever validation macro-F1 improves and breaks after `PATIENCE` stale epochs. Passing the phase-1 `history` into phase 2 keeps one continuous record and starts phase 2's early-stopping baseline from the phase-1 best.

In [ ]:
def run_phase(epochs, lr, tag, history=None):
    """One training phase. Checkpoints and early-stops on val macro-F1; writes best.pt."""
    history = history or {'epoch': [], 'phase': [], 'train_loss': [],
                          'val_loss': [], 'val_acc': [], 'val_f1': []}
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs, 1))
    best_f1 = max(history['val_f1']) if history['val_f1'] else -1.0
    stale = 0

    for epoch in range(1, epochs + 1):
        model.train()
        started = time.perf_counter()
        running_loss, seen = 0.0, 0
        for pre, post, labels in loaders['train']:
            pre = pre.to(device, non_blocking=True)
            post = post.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with autocast():
                loss = criterion(model(pre, post), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * labels.size(0)
            seen += labels.size(0)
        scheduler.step()

        val_loss, y_true, y_pred, _ = evaluate(loaders['val'])
        val_acc = float((y_true == y_pred).mean())
        val_f1 = float(f1_score(y_true, y_pred, average='macro', zero_division=0))
        history['epoch'].append(len(history['epoch']) + 1)
        history['phase'].append(tag)
        history['train_loss'].append(running_loss / max(seen, 1))
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)

        improved = val_f1 > best_f1
        if improved:
            best_f1, stale = val_f1, 0
            torch.save({'state_dict': model.state_dict(), 'arch': 'SiamesePretrained-resnet18',
                        'classes': CLASSES, 'val_macro_f1': best_f1,
                        'parameter_count': parameter_count}, best_path)
        else:
            stale += 1
        print(f"  [{tag}] epoch {epoch:02d}/{epochs} "
              f"train={history['train_loss'][-1]:.4f} val={val_loss:.4f} "
              f"acc={val_acc:.4f} macroF1={val_f1:.4f} ({time.perf_counter() - started:.0f}s)"
              f"{' <- best' if improved else ''}")
        if stale >= PATIENCE:
            print(f'  early stop: no val macro-F1 improvement for {PATIENCE} epochs')
            break
    return history


print('Training function ready.')

### 3.5 Run training and restore the best checkpoint

Phase 1 (frozen backbone) then phase 2 (fine-tune `layer3`/`layer4`). When both phases finish, `model` holds the best weights.

In [ ]:
print('PHASE 1 - frozen backbone, head only')
set_seed(SEED)
model.set_trainable(unfreeze=())
history = run_phase(EPOCHS_HEAD, LR_HEAD, 'head')

print('\nPHASE 2 - fine-tune layer3 + layer4')
model.set_trainable(unfreeze=('layer3', 'layer4'))
history = run_phase(EPOCHS_FT, LR_FT, 'finetune', history=history)

model.load_state_dict(torch.load(best_path, map_location=device)['state_dict'])
best_f1 = max(history['val_f1'])
best_epoch = int(np.argmax(history['val_f1']) + 1)
print(f'\nRestored best checkpoint: epoch={best_epoch}, val macro-F1={best_f1:.4f}')

## Task 4: Evaluation and Comparison

The restored best model is scored once on `val`, `test_id`, and `test_ood`. This section writes the metric artifacts, plots the learning curves and confusion matrices, and quantifies the in-distribution to wildfire transfer gap.

### 4.1 Held-out metrics and saved artifacts

`full_report` computes accuracy, macro-F1, weighted-F1, macro ROC-AUC, and per-class precision/recall/F1 for each split. Overall and per-class tables, confusion matrices, the run configuration, the training history, and the final checkpoint are all written under `work-pretrained/`.

In [ ]:
def full_report(y_true, y_pred, y_prob, name):
    accuracy = float((y_true == y_pred).mean())
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    try:
        auc = float(roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro'))
    except ValueError:
        auc = float('nan')
    precision, recall, per_f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=range(len(CLASSES)), zero_division=0)
    overall = {'model': 'pretrained_siamese_resnet18', 'split': name, 'n': len(y_true),
               'accuracy': round(accuracy, 4), 'macro_F1': round(macro_f1, 4),
               'weighted_F1': round(weighted_f1, 4), 'macro_ROC_AUC': round(auc, 4)}
    per_class = pd.DataFrame({'model': 'pretrained_siamese_resnet18', 'split': name,
                              'class': CLASSES, 'precision': precision, 'recall': recall,
                              'f1': per_f1, 'support': support})
    return overall, per_class


predictions = {}
overall_rows, per_class_frames, confusion_frames = [], [], []
for split in ['val', 'test_id', 'test_ood']:
    _, y_true, y_pred, y_prob = evaluate(loaders[split])
    predictions[split] = (y_true, y_pred, y_prob)
    overall, per_class = full_report(y_true, y_pred, y_prob, split)
    overall_rows.append(overall)
    per_class_frames.append(per_class)
    confusion_frames.append(
        pd.DataFrame(confusion_matrix(y_true, y_pred, labels=range(len(CLASSES))),
                     index=CLASSES, columns=CLASSES).rename_axis('true_class').reset_index()
          .assign(split=split, model='pretrained_siamese_resnet18'))

overall_df = pd.DataFrame(overall_rows)
per_class_df = pd.concat(per_class_frames, ignore_index=True)
overall_df.to_csv(OUTPUT_DIR / 'results_overall.csv', index=False)
per_class_df.to_csv(OUTPUT_DIR / 'results_per_class.csv', index=False)
pd.concat(confusion_frames, ignore_index=True).to_csv(OUTPUT_DIR / 'confusion_matrices.csv', index=False)

run_config = {'patch_dir': str(PATCH_DIR), 'output_dir': str(OUTPUT_DIR),
              'epochs_head': EPOCHS_HEAD, 'epochs_ft': EPOCHS_FT, 'batch_size': BATCH_SIZE,
              'lr_head': LR_HEAD, 'lr_ft': LR_FT, 'weight_decay': WEIGHT_DECAY,
              'patience': PATIENCE, 'seed': SEED, 'device': str(device),
              'gpu': torch.cuda.get_device_name(0) if amp_enabled else None,
              'pretrained': bool(model.pretrained), 'parameter_count': parameter_count,
              'best_epoch': best_epoch, 'best_val_macro_f1': float(best_f1)}
(OUTPUT_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')

torch.save({'state_dict': model.state_dict(), 'arch': 'SiamesePretrained-resnet18',
            'classes': CLASSES, 'val_macro_f1': float(best_f1),
            'parameter_count': parameter_count},
           checkpoint_dir / 'pretrained_siamese_resnet18.pt')
pd.DataFrame(history).to_csv(OUTPUT_DIR / 'training_history.csv', index=False)
display(overall_df)
print('Saved checkpoint:', checkpoint_dir / 'pretrained_siamese_resnet18.pt')
print('Saved artifacts :', OUTPUT_DIR)

### 4.2 Qualitative prediction check

Metrics say how often the model is right; these examples make its successes and mistakes inspectable. Each panel shows the pre-disaster patch on the left and the post-disaster patch on the right, with one correct and one incorrect prediction from each held-out test split.

In [ ]:
# One correct and one incorrect PRE | POST prediction from each held-out test split.
TEST_SPLITS = ['test_id', 'test_ood']
examples_by_split = {}
model.eval()
with torch.no_grad():
    for split in TEST_SPLITS:
        view_dataset = datasets[split]
        correct = incorrect = None
        offset = 0
        for pre, post, labels in loaders[split]:
            pre_device = pre.to(device, non_blocking=True)
            post_device = post.to(device, non_blocking=True)
            with autocast():
                preds = model(pre_device, post_device).argmax(1).cpu().numpy()
            for local_index, (true_label, predicted_label) in enumerate(zip(labels.numpy(), preds)):
                example = (offset + local_index, int(true_label), int(predicted_label))
                if true_label == predicted_label and correct is None:
                    correct = example
                elif true_label != predicted_label and incorrect is None:
                    incorrect = example
            offset += len(labels)
            if correct is not None and incorrect is not None:
                break
        examples_by_split[split] = (view_dataset, correct, incorrect)

fig, axes = plt.subplots(2, 2, figsize=(8, 8), squeeze=False)
for row_index, split in enumerate(TEST_SPLITS):
    view_dataset, correct, incorrect = examples_by_split[split]
    pre_images, post_images = view_dataset.arrays()
    for column_index, (kind, example, color) in enumerate(
            [('Correct', correct, 'seagreen'), ('Incorrect', incorrect, 'crimson')]):
        axis = axes[row_index, column_index]
        axis.axis('off')
        if example is None:
            axis.set_title(f'{split} — no {kind.lower()} example found', color=color)
            continue
        sample_index, true_label, predicted_label = example
        patch_row = view_dataset.rows[sample_index]
        separator = np.full((pre_images.shape[1], 3, 3), 255, np.uint8)
        pair = np.concatenate([np.asarray(pre_images[patch_row]), separator, np.asarray(post_images[patch_row])],
                              axis=1)
        axis.imshow(pair)
        axis.set_title(f'{split} — {kind}\ntrue: {CLASSES[true_label]} | pred: {CLASSES[predicted_label]}',
                       color=color, fontsize=9)
    axes[row_index, 0].set_ylabel(f'{split}\nPRE | POST', fontsize=10)
fig.suptitle('Best-model test predictions', y=.98)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'prediction_examples.png', dpi=160)
plt.show()

### 4.3 Learning curves

Train/validation loss, validation accuracy, and validation macro-F1 across both phases; the dotted line marks the phase-1 to phase-2 switch. Under a weighted loss on imbalanced data, validation loss can rise while macro-F1 keeps improving — which is why loss would be the wrong early-stopping signal here.

In [ ]:
history_frame = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history_frame.epoch, history_frame.train_loss, label='train')
axes[0].plot(history_frame.epoch, history_frame.val_loss, label='val')
axes[0].set(title='Loss', xlabel='epoch')
axes[0].legend()
axes[1].plot(history_frame.epoch, history_frame.val_acc, color='darkorange')
axes[1].set(title='Validation accuracy', xlabel='epoch')
axes[2].plot(history_frame.epoch, history_frame.val_f1, color='seagreen')
axes[2].set(title='Validation macro-F1', xlabel='epoch')
switch = int((history_frame.phase == 'head').sum()) + 0.5
for axis in axes:
    axis.grid(alpha=.25)
    axis.axvline(switch, color='grey', ls=':', lw=1)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'training_curves.png', dpi=160)
plt.show()

### 4.4 Per-class breakdown and confusion matrices

Per-class precision/recall/F1 and row-normalised confusion matrices (recall by true class) for the three splits. With `minor` and `major` making up under 1% each of the wildfire patches, the per-class view is the only honest way to read `test_ood`.

In [ ]:
print('Per-class metrics (overall accuracy hides the minority classes):')
display(per_class_df.pivot(index='class', columns='split',
                           values=['precision', 'recall', 'f1']).round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, split in zip(axes, ['val', 'test_id', 'test_ood']):
    y_true, y_pred, _ = predictions[split]
    matrix = confusion_matrix(y_true, y_pred, labels=range(len(CLASSES))).astype(float)
    normalized = matrix / matrix.sum(axis=1, keepdims=True).clip(min=1)
    axis.imshow(normalized, cmap='Blues', vmin=0, vmax=1)
    axis.set(xticks=range(len(CLASSES)), yticks=range(len(CLASSES)),
             xticklabels=SHORT, yticklabels=SHORT, xlabel='predicted', ylabel='true',
             title=f'{split} - recall by true class')
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            axis.text(j, i, f'{normalized[i, j]:.2f}', ha='center', va='center', fontsize=8,
                      color='white' if normalized[i, j] > 0.5 else 'black')
    axis.grid(False)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'confusion_matrices.png', dpi=160)
plt.show()

### 4.5 Test-ID versus wildfire test-OOD

The headline comparison: aggregate scores side by side, then the **direction** of the out-of-domain errors. A strong over-estimation asymmetry points to a shifted class prior — recalibratable without retraining — whereas diffuse, symmetric errors would mean the learned features themselves do not transfer.

In [ ]:
id_row = overall_df.set_index('split').loc['test_id']
ood_row = overall_df.set_index('split').loc['test_ood']
gap = pd.DataFrame({
    'test_id (seen types)': [id_row.accuracy, id_row.macro_F1, id_row.macro_ROC_AUC],
    'test_ood (wildfire, unseen)': [ood_row.accuracy, ood_row.macro_F1, ood_row.macro_ROC_AUC],
    'absolute drop': [round(id_row.accuracy - ood_row.accuracy, 4),
                      round(id_row.macro_F1 - ood_row.macro_F1, 4),
                      round(id_row.macro_ROC_AUC - ood_row.macro_ROC_AUC, 4)],
}, index=['accuracy', 'macro_F1', 'macro_ROC_AUC'])
print('CROSS-DISASTER TRANSFER GAP')
display(gap)

# Direction of the out-of-domain errors.
y_true, y_pred, _ = predictions['test_ood']
cm_ood = confusion_matrix(y_true, y_pred, labels=range(len(CLASSES)))
off_diagonal = cm_ood.sum() - np.trace(cm_ood)
over = np.triu(cm_ood, 1).sum()          # predicted more damage than truth
under = np.tril(cm_ood, -1).sum()        # predicted less damage than truth
print(f'\nOOD errors: {100 * over / max(off_diagonal, 1):.1f}% over-estimate severity, '
      f'{100 * under / max(off_diagonal, 1):.1f}% under-estimate.')
no_damage = cm_ood[0]
print(f'False alarms on true no-damage: {100 * no_damage[1:].sum() / max(no_damage.sum(), 1):.1f}% '
      f'of {no_damage.sum():,} undamaged wildfire buildings flagged as damaged.')

### 4.6 Discussion

Read the tables above together:

- **The gap is real, not a validation artefact.** `val` and `test_id` macro-F1 sit within a few thousandths of each other, so the drop to `test_ood` is a genuine domain effect rather than a lucky validation draw.
- **Macro-F1 overstates the failure.** The collapse concentrates in `minor` and `major`, which are ~10% of training patches but ~1% of wildfire patches — a base-rate effect that caps their precision regardless of feature quality. Check whether `destroyed` and `no-damage`, the classes that actually occur under wildfire, hold up; in the reference run they do, with `destroyed` F1 *higher* out of domain than in it.
- **ROC-AUC versus macro-F1.** If threshold-free ROC-AUC holds far better than macro-F1 out of domain, the class ranking survives and only the decision boundary is wrong — a recalibration problem, not a representation one. The error-direction asymmetry above is the corroborating signal.
- **Against the from-scratch model.** `train_scratch_cnn.ipynb` reports the same metrics on the same splits. A consistent edge here, especially on `test_ood`, is attributable to ImageNet pretraining because the data, splits, and evaluation are held fixed.

## Outputs to keep

Under `work-pretrained/`:

- `checkpoints/best.pt` and `checkpoints/pretrained_siamese_resnet18.pt` — the selected weights (by validation macro-F1)
- `results_overall.csv`, `results_per_class.csv`, `confusion_matrices.csv` — metrics for `val` / `test_id` / `test_ood`
- `training_history.csv`, `training_curves.png`, `confusion_matrices.png`, `prediction_examples.png` — training and error diagnostics
- `run_config.json` — every hyperparameter and the environment for this run